# Patient characteristics: sex and age

Does sex or age at surgery go with improvement? Observed counts first, then the Firth models (`improved ~ predictor`, one each). The cohort is every patient with the outcome, sex and age recorded, so it differs from the seizure-type cohort (`outcome_overview.ipynb`). Sex is coded 1 = male; age is per year at surgery.

The age test is Mann-Whitney, with the Hodges-Lehmann difference (improved minus not improved, in years) and its 95% CI; these match R's `wilcox.test(conf.int = TRUE)`. LGS is in `1_improvement/types/Firth_results.ipynb`. The number of seizure types before surgery is in `2_disappearance/types/count_change.ipynb`, alongside the number after and the number lost.

The models are fitted in R with `logistf`: `firths_covariates.R` writes `firths_results_covariates.csv`, and this notebook re-runs it. Odds ratios have profile-likelihood 95% CIs; Holm corrects across the two characteristics, within each outcome. The output goes to `results/<version>/0_cohort/`.

**Reading the results** (checked on v2, 2026-10-20): neither sex nor age at surgery goes with improvement on either outcome; the observed rates, the age medians and the Firth ORs all agree.

In [ ]:
import subprocess

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import fisher_exact, mannwhitneyu

from cc_stats.outcomes import OUTCOMES, heading
from cc_stats.paths import ANALYSIS_CSV, PROJECT_ROOT, results_dir
from cc_stats.plots import INK, PAIR_A, PAIR_B, forest, paired_rate_plot, right_of, save_figure, setup_or_axis
from cc_stats.rscript import find_rscript
from cc_stats.stats import hodges_lehmann

RSCRIPT = find_rscript()
print(f"Using {RSCRIPT}")
run = subprocess.run([RSCRIPT, "analysis/0_cohort/firths_covariates.R"], capture_output=True, text=True, cwd=PROJECT_ROOT)
if run.returncode != 0:
    raise RuntimeError(f"firths_covariates.R failed:\n{run.stderr}")
table = pd.read_csv(results_dir("0_cohort") / "firths_results_covariates.csv")
cov = {outcome: table.query("outcome == @outcome").reset_index(drop=True) for outcome in OUTCOMES}
analysis = pd.read_csv(ANALYSIS_CSV, index_col="subject_id")

In [ ]:
def fmt_p(v):
    return '<0.001' if v < 0.001 else f'{v:.3f}'

def fmt_num(v):
    return f'{v:.3f}' if v < 0.01 else f'{v:.2f}'

def fmt_or(o, lo, hi):
    return f'{fmt_num(o)} ({fmt_num(lo)}–{fmt_num(hi)})'


In [ ]:
covariates = {outcome: analysis.dropna(subset=[O['improved'], 'sex', 'age_at_cc']) for outcome, O in OUTCOMES.items()}
for outcome, c in covariates.items():
    print(f"{OUTCOMES[outcome]['name']}: n = {len(c)}, {int((c[OUTCOMES[outcome]['improved']] == 0).sum())} not improved")

In [ ]:
# Observed improvement by sex, one row per outcome
k = {sex: [] for sex in ('Male', 'Female')}
n = {sex: [] for sex in ('Male', 'Female')}
for outcome, O in OUTCOMES.items():
    c = covariates[outcome]
    for sex in k:
        group = c.loc[c.sex == sex, O['improved']]
        k[sex].append(int(group.sum()))
        n[sex].append(len(group))

fig, ax = plt.subplots(figsize=(9.5, 3))
# Fisher's exact test, male vs female, one per outcome (so no Holm)
p = [fisher_exact([[km, nm - km], [kf, nf - kf]])[1]
     for km, nm, kf, nf in zip(k['Male'], n['Male'], k['Female'], n['Female'])]
# Rows are drawn bottom up, so reverse them to put Engel on top
paired_rate_plot(ax, [f"{O['name']} ({O['improved_label']})" for O in OUTCOMES.values()][::-1],
                 top=('male', k['Male'][::-1], n['Male'][::-1], PAIR_A),
                 bottom=('female', k['Female'][::-1], n['Female'][::-1], PAIR_B), p=p[::-1])
ax.set_xlabel('Observed improvement (Wilson 95% CI)')
ax.set_title('Observed improvement by sex', loc='left', pad=28)
ax.legend(loc='lower left', bbox_to_anchor=(0, 1.0), ncol=2, frameon=False, handletextpad=0.3)
plt.tight_layout()
save_figure('0_cohort', 'observed_improvement_by_sex')
plt.show()

In [ ]:
# Age at surgery, improved vs not improved: Mann-Whitney and Hodges-Lehmann
rows = []
for outcome, O in OUTCOMES.items():
    c = covariates[outcome]
    age_improved = c.loc[c[O['improved']] == 1, 'age_at_cc']
    age_not_improved = c.loc[c[O['improved']] == 0, 'age_at_cc']
    hl, hl_lo, hl_hi = hodges_lehmann(age_improved, age_not_improved)
    rows.append({'Outcome': O['name'],
                 'Improved / not': f'{len(age_improved)} / {len(age_not_improved)}',
                 'Median age, improved': f'{age_improved.median():.0f}',
                 'Median age, not improved': f'{age_not_improved.median():.0f}',
                 'HL difference, years (95% CI)': f'{hl:+.1f} ({hl_lo:+.1f} to {hl_hi:+.1f})',
                 'Mann-Whitney p': fmt_p(mannwhitneyu(age_improved, age_not_improved).pvalue)})
display(pd.DataFrame(rows).style.hide(axis='index'))

fig, axes = plt.subplots(1, len(OUTCOMES), figsize=(11, 4.5), sharey=True)
rng = np.random.default_rng(7)
for ax, (outcome, O) in zip(axes, OUTCOMES.items()):
    c = covariates[outcome]
    groups = [('all', c['age_at_cc'], 'tab:gray'),
              ('improved', c.loc[c[O['improved']] == 1, 'age_at_cc'], PAIR_A),
              ('not improved', c.loc[c[O['improved']] == 0, 'age_at_cc'], PAIR_B)]
    for i, (name, v, color) in enumerate(groups):
        ax.boxplot(v, positions=[i], widths=0.45, showfliers=False, medianprops=dict(color=INK, lw=2))
        ax.scatter(i + rng.uniform(-0.12, 0.12, len(v)), v, s=20, color=color, alpha=0.7, zorder=3)
        ax.text(i, 1.0, f'n={len(v)}\nmedian {v.median():.0f}', transform=ax.get_xaxis_transform(),
                ha='center', va='bottom', fontsize=9)
    ax.set_xticks(range(3), [g[0] for g in groups])
    ax.set_ylim(0, None)
    mw_p = mannwhitneyu(groups[1][1], groups[2][1]).pvalue
    ax.set_title(f"{O['name']} ({O['improved_label']}): Mann-Whitney p = {fmt_p(mw_p)}", loc='left', pad=34)
    ax.spines[['top', 'right']].set_visible(False)
axes[0].set_ylabel('Age at surgery (years)')
plt.tight_layout()
save_figure('0_cohort', 'age_by_outcome')
plt.show()

In [ ]:
for outcome, O in OUTCOMES.items():
    heading(outcome)
    c = cov[outcome].sort_values("p").reset_index(drop=True)
    display(pd.DataFrame({
        'Predictor': c.predictor,
        'Mean, improved': c.mean_improved.map(lambda v: f'{v:.2f}'),
        'Mean, not improved': c.mean_not_improved.map(lambda v: f'{v:.2f}'),
        'OR (95% CI)': [fmt_or(*r) for r in c[['OR', 'lo', 'hi']].to_numpy()],
        'p': c.p.map(fmt_p),
        'Holm p': c.p_holm.map(fmt_p),
    }).style.hide(axis='index'))

In [ ]:
# Both outcomes on one plot: per characteristic, Engel (teal, above) and ILAE
# (orange, below), with Holm p for each in a headed column.
names = {'sex': 'sex (male)', 'age_at_cc': 'age at surgery (per year)'}
order = ['age_at_cc', 'sex']  # bottom to top
fig, ax = plt.subplots(figsize=(9, 3.4))
for (outcome, O), offset in zip(OUTCOMES.items(), (0.17, -0.17)):
    d = cov[outcome].set_index('predictor').loc[order].reset_index()
    forest(ax, d, ['OR', 'lo', 'hi'], offset=offset, color={'engel': PAIR_A, 'ilae': PAIR_B}[outcome], label=O['name'])
setup_or_axis(ax, [names[p] for p in order], '', ticks=(0.1, 0.25, 0.5, 1, 2, 4, 10))
ax.set_ylim(-0.6, len(order) - 0.4)
ax.set_title('Patient characteristics, univariable Firth models', loc='left', pad=26)
# Columns a fixed distance (inches) right of the plot: (centre, header)
columns = [(0.45, 'Outcome'), (1.45, 'n'), (2.45, 'Holm p')]
for x, header in columns:
    ax.text(1, len(order) - 0.3, header, transform=right_of(ax, x), va='center', ha='center',
            fontsize=9, fontweight='bold')
for (outcome, O), offset in zip(OUTCOMES.items(), (0.17, -0.17)):
    d = cov[outcome].set_index('predictor')
    c = covariates[outcome]
    n_text = {'sex': f"{int((c.sex == 'Male').sum())} male / {int((c.sex == 'Female').sum())} female",
              'age_at_cc': f'{len(c)}'}
    for y, p in enumerate(order):
        r = d.loc[p]
        for (x, _), value in zip(columns, (O['name'], n_text[p], fmt_p(r.p_holm))):
            ax.text(1, y + offset, value, transform=right_of(ax, x), va='center', ha='center', fontsize=9,
                    color={'engel': PAIR_A, 'ilae': PAIR_B}[outcome])
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.3), ncol=2, frameon=False)
plt.tight_layout()
save_figure('0_cohort', 'firth_patient_characteristics')
plt.show()